In [ ]:
# %pip install tifffile torch numpy scipy scikit-image matplotlib pandas scikit-learn tabulate --quiet


In [ ]:
import os
import sys
import re
import json
import time
import copy
import string
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

%matplotlib inline


In [ ]:
OUT_DIR_BASE    = "D:/jupyter/Latent Spacing/NewNew_Output_modifiedLatest"
REGISTRY_PATH   = os.path.join(OUT_DIR_BASE, "dataset_registry.json")
CHECKPOINT_PATH = os.path.join(OUT_DIR_BASE, "vae_checkpoint.pt")
TRAINING_LOG_PATH = os.path.join(OUT_DIR_BASE, "training_log.json")
os.makedirs(OUT_DIR_BASE, exist_ok=True)

DET_SIZE = 512                  # confirmed identical across all 3 (same detector)
MIB_DTYPE = np.dtype(">u2")     # flip to "<u2" if sanity-check patterns look garbled


NEW_DATASETS = [
    {"name": "Al_1", "mib_path": "D:/jupyter/Latent Spacing/Data/20250324 170958/Al_1.mib", "hdr_path": "D:/jupyter/Latent Spacing/Data/20250324 170958/Al_1.hdr"},
]


In [ ]:
def parse_hdr_frame_count(hdr_path):
    with open(hdr_path, "r", errors="ignore") as f:
        text = f.read()
    m = re.search(r"Frames in Acquisition \(Number\):\s*(\d+)", text)
    if not m:
        raise ValueError(f"Could not find frame count in {hdr_path}")
    return int(m.group(1))


def infer_scan_shape(hdr_path, manual_ny=None, manual_nx=None):
    if manual_ny is not None and manual_nx is not None:
        return manual_ny, manual_nx
    n_frames = parse_hdr_frame_count(hdr_path)
    sqrt_n = int(round(n_frames ** 0.5))
    if sqrt_n * sqrt_n != n_frames:
        raise ValueError(
            f"{hdr_path}: {n_frames} frames is not a perfect square -- this scan isn't "
            f"square. Pass manual_ny/manual_nx explicitly for this dataset."
        )
    return sqrt_n, sqrt_n


In [ ]:
def diagnose_mib_layout(mib_path, det_size, dtype_itemsize, header_range=(0, 2000)):
  
    filesize = os.path.getsize(mib_path)
    frame_bytes = det_size * det_size * dtype_itemsize
    lines = [f"Actual file size: {filesize} bytes ({filesize/1e9:.3f} GB)",
              f"Solving for header_bytes in [{header_range[0]}, {header_range[1]}] "
              f"at det={det_size}, dtype_bytes={dtype_itemsize}:", ""]
    found = []
    for header_bytes in range(header_range[0], header_range[1] + 1):
        record_bytes = frame_bytes + header_bytes
        if record_bytes == 0 or filesize % record_bytes != 0:
            continue
        n_frames = filesize // record_bytes
        sqrt_n = int(round(n_frames ** 0.5))
        is_square = sqrt_n * sqrt_n == n_frames
        if is_square and n_frames > 0:
            found.append((header_bytes, n_frames, sqrt_n))
            lines.append(f"  header_bytes={header_bytes:5d}  ->  n_frames={n_frames:8d}  "
                          f"({sqrt_n}x{sqrt_n} scan)  <-- PLAUSIBLE (perfect square)")
    if not found:
        lines.append("  No header size in this range gives a perfect-square frame count at "
                      "this det_size/dtype -- try a different det_size/dtype, or the file may "
                      "genuinely be a different detector configuration or incomplete.")
    return chr(10).join(lines), found


def validate_mib_layout(mib_path, ny, nx, det_size, dtype_itemsize):
   
    filesize = os.path.getsize(mib_path)
    n_frames = ny * nx
    frame_bytes = det_size * det_size * dtype_itemsize
    record_bytes = filesize / n_frames
    header_bytes = record_bytes - frame_bytes

    if header_bytes < 0 or abs(header_bytes - round(header_bytes)) > 1e-6 or header_bytes > 2000:
        diag_text, found = diagnose_mib_layout(mib_path, det_size, dtype_itemsize)
        msg = (
            f"\nGEOMETRY MISMATCH for '{mib_path}'\n"
            f"Assumed {ny}x{nx} scan ({n_frames} frames) at {det_size}x{det_size} detector "
            f"gives an implausible header size ({header_bytes:.2f} bytes).\n"
            f"This usually means the .hdr's frame count doesn't match what's actually on "
            f"disk (scan stopped early, wrong file, or an incomplete copy).\n\n{diag_text}\n"
        )
        if found:
            msg += (f"\nSolved candidate(s) -- (header_bytes, n_frames, scan_side): {found}\n"
                     f"Set this dataset's manual_ny/manual_nx to the scan_side value above "
                     f"before re-adding.\n")
        else:
            msg += "\nTry re-running diagnose_mib_layout() with a different det_size/dtype_itemsize.\n"
        raise ValueError(msg)
    return True


def load_registry(path):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return []


def save_registry(path, registry):
    with open(path, "w") as f:
        json.dump(registry, f, indent=2)


def merge_new_datasets(registry, new_datasets, det_size, dtype_itemsize):
    existing_names = {d["name"] for d in registry}
    for ds in new_datasets:
        if ds["name"] in existing_names:
            print(f"'{ds['name']}' already in registry -- skipping (remove/rename to re-add).")
            continue
        ny, nx = infer_scan_shape(ds["hdr_path"], ds.get("manual_ny"), ds.get("manual_nx"))
        validate_mib_layout(ds["mib_path"], ny, nx, det_size, dtype_itemsize)  # raises with a clear diagnostic if wrong
        entry = {"name": ds["name"], "mib_path": ds["mib_path"], "hdr_path": ds["hdr_path"],
                 "scan_ny": ny, "scan_nx": nx}
        registry.append(entry)
        print(f"Added '{ds['name']}' to registry: scan {ny}x{nx} (geometry validated against actual file size)")
    return registry


In [ ]:
registry = load_registry(REGISTRY_PATH)
registry = merge_new_datasets(registry, NEW_DATASETS, DET_SIZE, MIB_DTYPE.itemsize)
save_registry(REGISTRY_PATH, registry)

print(f"\nFull registry ({len(registry)} datasets):")
for d in registry:
    print(f"  {d['name']}: scan {d['scan_ny']}x{d['scan_nx']}, mib={d['mib_path']}")


In [ ]:
class MibFastReader:
    def __init__(self, mib_path, ny, nx, det_size=512, dtype=np.dtype(">u2")):
        self.mib_path = mib_path
        self.ny, self.nx, self.det_size, self.dtype = ny, nx, det_size, dtype
        n_frames = ny * nx
        filesize = os.path.getsize(mib_path)
        frame_bytes = det_size * det_size * dtype.itemsize
        record_bytes = filesize / n_frames
        header_bytes = record_bytes - frame_bytes
        assert header_bytes == int(header_bytes), (
            f"Non-integer header size ({header_bytes}) for {mib_path} -- "
            f"scan size {ny}x{nx} doesn't match this file's actual frame count."
        )
        self.header_bytes = int(header_bytes)
        self.record_bytes = int(record_bytes)
        self.frame_bytes = frame_bytes
        self._mm = None
        print(f"MibFastReader[{os.path.basename(mib_path)}]: {ny}x{nx} scan, "
              f"header={self.header_bytes}B, n_frames={n_frames}")

    def _mmap(self):
        if self._mm is None:
            self._mm = np.memmap(self.mib_path, dtype=np.uint8, mode="r")
        return self._mm

    def get_frame(self, iy, ix):
        idx = iy * self.nx + ix
        mm = self._mmap()
        start = idx * self.record_bytes + self.header_bytes
        end = start + self.frame_bytes
        raw = mm[start:end].view(self.dtype).reshape(self.det_size, self.det_size)
        return raw.astype(np.float32)

    def get_row(self, iy):
        return np.stack([self.get_frame(iy, ix) for ix in range(self.nx)])


In [ ]:
readers = {}
for d in registry:
    readers[d["name"]] = MibFastReader(d["mib_path"], d["scan_ny"], d["scan_nx"], det_size=DET_SIZE, dtype=MIB_DTYPE)
    os.makedirs(os.path.join(OUT_DIR_BASE, d["name"]), exist_ok=True)

fig, axes = plt.subplots(1, len(registry), figsize=(4*len(registry), 4))
if len(registry) == 1:
    axes = [axes]
for ax, d in zip(axes, registry):
    r = readers[d["name"]]
    frame = r.get_frame(r.ny // 2, r.nx // 2)
    ax.imshow(np.log1p(frame), cmap="inferno")
    ax.set_title(f"{d['name']} ({r.ny}x{r.nx})")
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import laplace

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

BINNING           = 4
LATENT_DIM        = 128
BASE_CH           = 32
KERNEL_SIZE       = 5
BATCH_SIZE        = 256
LR                = 1e-3
ADAM_BETAS        = (0.9, 0.999)
BETA_KL           = 1.0
TRAIN_FRACTION    = 0.8
N_TRAIN_PER_DATASET = 20000  # only one dataset now
SEED              = 0

IS_FIRST_EVER_RUN = not os.path.exists(CHECKPOINT_PATH)   # auto-detected
N_EPOCHS          = 250 if IS_FIRST_EVER_RUN else 60
KL_WARMUP_EPOCHS  = 30 if IS_FIRST_EVER_RUN else 3
PATIENCE          = 15 if IS_FIRST_EVER_RUN else 10

print(f"{'First-ever pooled training run' if IS_FIRST_EVER_RUN else 'Incremental fine-tuning round'} "
      f"-- N_EPOCHS={N_EPOCHS}, KL_WARMUP_EPOCHS={KL_WARMUP_EPOCHS}")

torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
def compute_mean_background(mib_reader, n_sample=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    iys = rng.integers(0, mib_reader.ny, n_sample)
    ixs = rng.integers(0, mib_reader.nx, n_sample)
    acc = np.zeros((mib_reader.det_size, mib_reader.det_size), dtype=np.float64)
    for iy, ix in zip(iys, ixs):
        acc += mib_reader.get_frame(iy, ix)
    return (acc / n_sample).astype(np.float32)


def make_active_mask(shape, direct_beam_radius_frac=0.03):
    dy, dx = shape
    yy, xx = np.ogrid[:dy, :dx]
    cy, cx = dy / 2, dx / 2
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    mask = np.ones(shape, dtype=np.float32)
    mask[r <= direct_beam_radius_frac * dy] = 0.0
    return mask


def preprocess_pattern(pattern, background, binning=BINNING, active_mask=None):
    """Background subtraction, binning, percentile clip + rescale to [-1,1].
    Deliberately NO augmentation -- preserves the raw physical content exactly."""
    p = pattern.astype(np.float32) - background
    dy, dx = p.shape
    if binning > 1:
        p = p[: dy - dy % binning, : dx - dx % binning]
        p = p.reshape(p.shape[0] // binning, binning, p.shape[1] // binning, binning).mean(axis=(1, 3))
    lo, hi = np.percentile(p, 0.5), np.percentile(p, 99.5)
    p = np.clip(p, lo, hi)
    p = 2 * (p - lo) / (hi - lo + 1e-8) - 1.0
    if active_mask is not None:
        p = p * active_mask
    return p.astype(np.float32)


In [ ]:
backgrounds = {}
for d in registry:
    print(f"Estimating background: {d['name']}")
    backgrounds[d["name"]] = compute_mean_background(readers[d["name"]])

BINNED_SIZE = DET_SIZE // BINNING
active_mask = make_active_mask((BINNED_SIZE, BINNED_SIZE))
active_mask_t = torch.from_numpy(active_mask).to(DEVICE)


In [ ]:
dataset = registry[0]["name"]
reader = readers[dataset]

frame = reader.get_frame(reader.ny//2, reader.nx//2)

background = backgrounds[dataset]

frame_bs = frame.astype(np.float32) - background

frame_pre = preprocess_pattern(frame, background, active_mask=None)

frame_masked = preprocess_pattern(frame, background,
                                  active_mask=active_mask)

fig,ax = plt.subplots(1,4,figsize=(18,5))

ax[0].imshow(np.log1p(frame),cmap='gray')
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(frame_bs,cmap='gray')
ax[1].set_title("Background subtracted")
ax[1].axis('off')

ax[3].imshow(frame_masked,cmap='gray',vmin=-1,vmax=1)
ax[3].set_title("After preprocessing")
ax[3].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
class PooledDiffractionPatternDataset(Dataset):
    def __init__(self, readers, backgrounds, active_mask, samples):
        self.readers = readers
        self.backgrounds = backgrounds
        self.active_mask = active_mask
        self.samples = samples  # list of (name, iy, ix)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name, iy, ix = self.samples[idx]
        raw = self.readers[name].get_frame(iy, ix)
        p = preprocess_pattern(raw, self.backgrounds[name], active_mask=self.active_mask)
        return torch.from_numpy(p).unsqueeze(0), name, (iy, ix)


In [ ]:
rng = np.random.default_rng(SEED)
train_samples, val_samples = [], []
for d in registry:
    name = d["name"]
    ny, nx = d["scan_ny"], d["scan_nx"]
    all_coords = [(iy, ix) for iy in range(ny) for ix in range(nx)]
    chosen_idx = rng.choice(len(all_coords), size=min(N_TRAIN_PER_DATASET, len(all_coords)), replace=False)
    chosen = [all_coords[i] for i in chosen_idx]
    n_train = int(TRAIN_FRACTION * len(chosen))
    train_samples += [(name, iy, ix) for iy, ix in chosen[:n_train]]
    val_samples += [(name, iy, ix) for iy, ix in chosen[n_train:]]

rng.shuffle(train_samples)
rng.shuffle(val_samples)

train_ds = PooledDiffractionPatternDataset(readers, backgrounds, active_mask, train_samples)
val_ds = PooledDiffractionPatternDataset(readers, backgrounds, active_mask, val_samples)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Pooled train patterns: {len(train_ds)} across {len(registry)} datasets: {[d['name'] for d in registry]}")
print(f"Pooled val patterns:   {len(val_ds)}")


In [ ]:
t0 = time.time()
batch = next(iter(train_loader))
print("time to fetch ONE batch:", time.time() - t0, "s   batch shape:", batch[0].shape)


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out, kernel=9):
        super().__init__()
        pad = kernel // 2
        self.down = nn.Conv2d(c_in, c_out, kernel, stride=2, padding=pad)
        self.res = nn.Conv2d(c_out, c_out, kernel, stride=1, padding=pad)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x = self.act(self.down(x))
        return self.act(self.res(x) + x)


class DeconvBlock(nn.Module):
    def __init__(self, c_in, c_out, kernel=9):
        super().__init__()
        pad = kernel // 2
        self.up = nn.ConvTranspose2d(c_in, c_out, kernel, stride=2, padding=pad, output_padding=1)
        self.res = nn.Conv2d(c_out, c_out, kernel, stride=1, padding=pad)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x = self.act(self.up(x))
        return self.act(self.res(x) + x)


class Encoder(nn.Module):
    def __init__(self, in_size, latent_dim, base_ch, kernel=9):
        super().__init__()
        self.block1 = ConvBlock(1, base_ch, kernel)
        self.block2 = ConvBlock(base_ch, base_ch * 2, kernel)
        self.block3 = ConvBlock(base_ch * 2, base_ch * 4, kernel)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc_mu = nn.Linear(base_ch * 4, latent_dim)
        self.fc_logvar = nn.Linear(base_ch * 4, latent_dim)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.pool(x).flatten(1)
        return self.fc_mu(x), self.fc_logvar(x)


class Decoder(nn.Module):
    def __init__(self, out_size, latent_dim, base_ch, kernel=9):
        super().__init__()
        self.init_size = out_size // 8
        self.base_ch = base_ch * 4
        self.fc = nn.Linear(latent_dim, self.base_ch * self.init_size * self.init_size)
        self.block1 = DeconvBlock(self.base_ch, base_ch * 2, kernel)
        self.block2 = DeconvBlock(base_ch * 2, base_ch, kernel)
        self.block3 = DeconvBlock(base_ch, base_ch, kernel)
        self.out_conv = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.out_act = nn.Tanh()

    def forward(self, z):
        x = self.fc(z).view(-1, self.base_ch, self.init_size, self.init_size)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.out_conv(x)
        return self.out_act(x)


class DiffractionVAE(nn.Module):
    def __init__(self, in_size, latent_dim, base_ch=BASE_CH, kernel=KERNEL_SIZE):
        super().__init__()
        self.base_ch = base_ch
        self.encoder = Encoder(in_size, latent_dim, base_ch, kernel)
        self.decoder = Decoder(in_size, latent_dim, base_ch, kernel)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar, z


def vae_loss(recon, target, mu, logvar, active_mask=None, beta=BETA_KL):
    if active_mask is not None:
        diff2 = (recon - target) ** 2 * active_mask
        l2 = diff2.sum() / target.shape[0]
    else:
        l2 = F.mse_loss(recon, target, reduction="sum") / target.shape[0]
    kld = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    return l2 + beta * kld, l2.item(), kld.item()


In [ ]:
model = DiffractionVAE(in_size=BINNED_SIZE, latent_dim=LATENT_DIM).to(DEVICE)

if os.path.exists(CHECKPOINT_PATH):
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
    print(f"Loaded existing checkpoint from {CHECKPOINT_PATH} -- continuing training.")
else:
    print("No existing checkpoint found- new run")

print("Total parameters:", sum(p.numel() for p in model.parameters()))


In [ ]:
def kl_beta(epoch, target_beta, warmup_epochs):
    if warmup_epochs <= 0:
        return target_beta
    return target_beta * min(1.0, epoch / warmup_epochs)


def train_vae(model, train_loader, val_loader, active_mask_t, n_epochs,
              patience=15, min_delta_frac=1e-3, restore_best_weights=True, bar_width=30,
              target_beta=BETA_KL, kl_warmup_epochs=30):
    opt = torch.optim.Adam(model.parameters(), lr=LR, betas=ADAM_BETAS)
    history = {"train_l2": [], "val_l2": [], "train_kld": [], "val_kld": [], "beta": [], "mu_std": []}

    n_train_samples = len(train_loader.dataset)
    n_train_batches = len(train_loader)
    best_val_l2 = float("inf")
    best_state = None
    epochs_no_improve = 0

    for epoch in range(n_epochs):
        beta = kl_beta(epoch, target_beta, kl_warmup_epochs)
        print(f"Epoch {epoch + 1}/{n_epochs}  (beta={beta:.3f})")
        model.train()
        tr_l2, tr_kld, n_seen = 0.0, 0.0, 0
        mu_stds = []
        epoch_t0 = time.time()

        for batch_idx, (x, _, _) in enumerate(train_loader):
            x = x.to(DEVICE)
            recon, mu, logvar, _ = model(x)
            loss, l2, kld = vae_loss(recon, x, mu, logvar, active_mask_t, beta=beta)
            opt.zero_grad()
            loss.backward()
            opt.step()

            bs = x.size(0)
            tr_l2 += l2 * bs; tr_kld += kld * bs; n_seen += bs
            mu_stds.append(mu.detach().std().item())

            elapsed = time.time() - epoch_t0
            us_per_step = (elapsed / (batch_idx + 1)) * 1e6
            frac = (batch_idx + 1) / n_train_batches
            filled = int(bar_width * frac)
            bar = "=" * bar_width if filled >= bar_width else "=" * filled + ">" + "." * (bar_width - filled - 1)
            sys.stdout.write(f"\r{n_seen}/{n_train_samples} [{bar}] - {elapsed:.0f}s "
                              f"{us_per_step:.0f}us/step - loss: {tr_l2 / n_seen:.4f}")
            sys.stdout.flush()

        history["train_l2"].append(tr_l2 / n_seen)
        history["train_kld"].append(tr_kld / n_seen)
        history["beta"].append(beta)
        history["mu_std"].append(sum(mu_stds) / len(mu_stds))

        model.eval()
        va_l2, va_kld, nv = 0.0, 0.0, 0
        with torch.no_grad():
            for x, _, _ in val_loader:
                x = x.to(DEVICE)
                recon, mu, logvar, _ = model(x)
                _, l2, kld = vae_loss(recon, x, mu, logvar, active_mask_t, beta=beta)
                bs = x.size(0)
                va_l2 += l2 * bs; va_kld += kld * bs; nv += bs
        history["val_l2"].append(va_l2 / nv)
        history["val_kld"].append(va_kld / nv)

        epoch_time = time.time() - epoch_t0
        us_per_step = (epoch_time / n_train_batches) * 1e6
        sys.stdout.write(f"\r{n_train_samples}/{n_train_samples} [{'=' * bar_width}] - {epoch_time:.0f}s "
                          f"{us_per_step:.0f}us/step - loss: {history['train_l2'][-1]:.4f} "
                          f"- val_loss: {history['val_l2'][-1]:.4f} - mu_std: {history['mu_std'][-1]:.4f}\n")
        sys.stdout.flush()

        if epoch >= kl_warmup_epochs:
            current_val_l2 = history["val_l2"][-1]
            if current_val_l2 < best_val_l2 * (1 - min_delta_frac):
                best_val_l2 = current_val_l2
                epochs_no_improve = 0
                if restore_best_weights:
                    best_state = copy.deepcopy(model.state_dict())
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f"Early stopping at epoch {epoch + 1} (best val_loss = {best_val_l2:.4f}).")
                    break

    if restore_best_weights and best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best model weights (val_loss = {best_val_l2:.4f}).")

    return history


In [ ]:
training_t0 = time.time()
history = train_vae(model, train_loader, val_loader, active_mask_t, n_epochs=N_EPOCHS,
                     patience=PATIENCE, kl_warmup_epochs=KL_WARMUP_EPOCHS)
TRAINING_TIME_SEC = time.time() - training_t0

torch.save(model.state_dict(), CHECKPOINT_PATH)
print(f"Checkpoint saved to {CHECKPOINT_PATH}")

log = []
if os.path.exists(TRAINING_LOG_PATH):
    with open(TRAINING_LOG_PATH, "r") as f:
        log = json.load(f)
log.append({
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "datasets": [d["name"] for d in registry],
    "n_epochs_run": len(history["train_l2"]),
    "final_train_l2": history["train_l2"][-1],
    "final_val_l2": history["val_l2"][-1],
    "final_mu_std": history["mu_std"][-1],
    "training_time_sec": TRAINING_TIME_SEC,
    "was_first_ever_run": IS_FIRST_EVER_RUN,
})
with open(TRAINING_LOG_PATH, "w") as f:
    json.dump(log, f, indent=2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history["train_l2"], label="train"); axes[0].plot(history["val_l2"], label="val")
axes[0].set_title("L2 reconstruction loss"); axes[0].set_yscale("log"); axes[0].legend()
axes[1].plot(history["train_kld"], label="train"); axes[1].plot(history["val_kld"], label="val")
axes[1].set_title("KL divergence"); axes[1].legend()
axes[2].plot(history["mu_std"]); axes[2].set_title("Latent mu std (collapse indicator)"); axes[2].set_xlabel("epoch")
plt.tight_layout()
plt.show()


In [ ]:
model.eval()
fig, axes = plt.subplots(2, len(registry), figsize=(4*len(registry), 8))
if len(registry) == 1:
    axes = axes.reshape(2, 1)
with torch.no_grad():
    for j, d in enumerate(registry):
        name = d["name"]
        r = readers[name]
        raw = r.get_frame(r.ny // 2, r.nx // 2)
        p = preprocess_pattern(raw, backgrounds[name], active_mask=active_mask)
        x = torch.from_numpy(p).unsqueeze(0).unsqueeze(0).to(DEVICE)
        recon, mu, logvar, z = model(x)
        axes[0, j].imshow(p, cmap="gray"); axes[0, j].set_title(f"{name} original"); axes[0, j].axis("off")
        axes[1, j].imshow(recon[0,0].cpu().numpy(), cmap="gray"); axes[1, j].set_title(f"{name} reconstructed"); axes[1, j].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
def make_quicklook(mib_reader, bf_radius_frac=0.08):
    ny, nx, dy, dx = mib_reader.ny, mib_reader.nx, mib_reader.det_size, mib_reader.det_size
    yy, xx = np.ogrid[:dy, :dx]
    cy, cx = dy / 2, dx / 2
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    bf_mask = r <= bf_radius_frac * dy
    adf_mask = ~bf_mask
    vbf = np.zeros((ny, nx), dtype=np.float64)
    vadf = np.zeros((ny, nx), dtype=np.float64)
    t0 = time.time()
    for iy in range(ny):
        row = mib_reader.get_row(iy)
        vbf[iy] = row[:, bf_mask].sum(axis=1)
        vadf[iy] = row[:, adf_mask].sum(axis=1)
        if iy % 32 == 0:
            print(f"    quicklook row {iy}/{ny}  ({time.time()-t0:.0f}s)", flush=True)
    return vbf, vadf


@torch.no_grad()
def encode_full_scan(model, mib_reader, background, active_mask, latent_dim):
    model.eval()
    ny, nx = mib_reader.ny, mib_reader.nx
    latent_maps = np.zeros((ny, nx, latent_dim), dtype=np.float32)
    t0 = time.time()
    for iy in range(ny):
        row_raw = mib_reader.get_row(iy)
        row_proc = np.stack([preprocess_pattern(row_raw[ix], background, active_mask=active_mask) for ix in range(nx)])
        x = torch.from_numpy(row_proc).unsqueeze(1).to(DEVICE)
        mu, _ = model.encoder(x)
        latent_maps[iy] = mu.cpu().numpy()
        if iy % 32 == 0:
            print(f"    row {iy}/{ny}  ({time.time()-t0:.0f}s)", flush=True)
    return latent_maps


def disk_center_of_mass(pattern, yy, xx):
    total = pattern.sum() + 1e-8
    return (pattern * yy).sum() / total, (pattern * xx).sum() / total


def pattern_sharpness(pattern):
    return laplace(pattern.astype(np.float64)).var()


def build_classical_maps(mib_reader, background=None, binning=1):
    ny, nx, det_size = mib_reader.ny, mib_reader.nx, mib_reader.det_size
    yy, xx = np.mgrid[0:det_size, 0:det_size]
    vbf = np.zeros((ny, nx)); com_y = np.zeros((ny, nx)); com_x = np.zeros((ny, nx)); sharp = np.zeros((ny, nx))
    t0 = time.time()
    for iy in range(ny):
        row = mib_reader.get_row(iy)
        for ix in range(nx):
            p_raw = row[ix].astype(np.float64)
            vbf[iy, ix] = p_raw.sum()
            cy, cx = disk_center_of_mass(p_raw, yy, xx)
            com_y[iy, ix] = cy; com_x[iy, ix] = cx
            p_sharp = p_raw - background if background is not None else p_raw
            if binning > 1:
                dy, dx = p_sharp.shape
                p_sharp = p_sharp[: dy - dy % binning, : dx - dx % binning]
                p_sharp = p_sharp.reshape(dy // binning, binning, dx // binning, binning).mean(axis=(1, 3))
            sharp[iy, ix] = pattern_sharpness(p_sharp)
        if iy % 8 == 0:
            print(f"    row {iy}/{ny}  ({time.time()-t0:.0f}s)", flush=True)
    return {"virtual_BF": vbf, "COM_y": com_y, "COM_x": com_x, "sharpness": sharp}


def select_diverse_latent_dims(latent_maps, top_k=6, corr_threshold=0.75, candidate_pool=40):
    flat = latent_maps.reshape(-1, latent_maps.shape[-1])
    variances = flat.var(axis=0)
    order = np.argsort(variances)[::-1][:candidate_pool]
    selected = []
    for dim in order:
        if len(selected) >= top_k:
            break
        if all(abs(np.corrcoef(flat[:, dim], flat[:, s])[0, 1]) <= corr_threshold for s in selected):
            selected.append(dim)
    return np.array(selected), variances[selected]


def robust_imshow(ax, arr, cmap="gray", p_lo=1, p_hi=99, title=None):
    vmin, vmax = np.percentile(arr, p_lo), np.percentile(arr, p_hi)
    ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    if title:
        ax.set_title(title)
    ax.axis("off")


In [ ]:
results = {}
for d in registry:
    name = d["name"]
    out_dir = os.path.join(OUT_DIR_BASE, name)

    print(f"=== Quicklook: {name} ===")
    vbf_q, vadf_q = make_quicklook(readers[name])
    tifffile.imwrite(os.path.join(out_dir, "virtual_BF_quicklook.tiff"), vbf_q.astype(np.float32))
    tifffile.imwrite(os.path.join(out_dir, "virtual_ADF_quicklook.tiff"), vadf_q.astype(np.float32))

    print(f"\n=== Encoding {name} ({d['scan_ny']}x{d['scan_nx']}) ===")
    latent_maps = encode_full_scan(model, readers[name], backgrounds[name], active_mask, LATENT_DIM)
    np.save(os.path.join(out_dir, f"latent_maps_{LATENT_DIM}d.npy"), latent_maps)

    print(f"=== Classical maps: {name} ===")
    classical_maps = build_classical_maps(readers[name], background=backgrounds[name], binning=BINNING)
    for map_name, arr in classical_maps.items():
        tifffile.imwrite(os.path.join(out_dir, f"classical_{map_name}.tiff"), arr.astype(np.float32))

    top_dims, top_vars = select_diverse_latent_dims(latent_maps, top_k=6, corr_threshold=0.75)
    print(f"{name}: found {len(top_dims)} diverse latent dims: {top_dims.tolist()}")
    results[name] = {"latent_maps": latent_maps, "classical_maps": classical_maps,
                      "top_dims": top_dims, "vbf_quicklook": vbf_q, "vadf_quicklook": vadf_q}


In [ ]:
for d in registry:
    name = d["name"]
    latent_maps = results[name]["latent_maps"]
    classical_maps = results[name]["classical_maps"]
    top_dims = results[name]["top_dims"]

    n_latent = len(top_dims)
    n_classical = len(classical_maps)
    fig, axes = plt.subplots(1, n_latent + n_classical, figsize=(3 * (n_latent + n_classical), 3))
    if n_latent + n_classical == 1:
        axes = [axes]
    for i, dim in enumerate(top_dims):
        robust_imshow(axes[i], latent_maps[:, :, dim], title=f"latent #{dim}")
    for j, (map_name, arr) in enumerate(classical_maps.items()):
        robust_imshow(axes[n_latent + j], arr, title=map_name)
    fig.suptitle(name)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR_BASE, name, "comparison_figure.png"), dpi=200)
    plt.show()


In [ ]:
def compute_latent_physical_correlations(latent_maps, classical_maps):
    ny, nx, latent_dim = latent_maps.shape
    flat_latent = latent_maps.reshape(-1, latent_dim)
    physical_names = list(classical_maps.keys())
    flat_physical = np.stack([classical_maps[name].ravel() for name in physical_names], axis=1)

    def standardize(a):
        mu = a.mean(axis=0, keepdims=True)
        sd = a.std(axis=0, keepdims=True) + 1e-12
        return (a - mu) / sd

    Z_latent = standardize(flat_latent)
    Z_physical = standardize(flat_physical)
    n = flat_latent.shape[0]
    corr_matrix = (Z_physical.T @ Z_latent) / n
    return corr_matrix, physical_names


def plot_correlation_heatmap(corr_matrix, physical_names, out_path=None, top_k_dims=40, dpi=200):
    max_abs = np.abs(corr_matrix).max(axis=0)
    top_dims = np.sort(np.argsort(max_abs)[::-1][:top_k_dims])
    sub = corr_matrix[:, top_dims]
    fig, ax = plt.subplots(figsize=(max(8, len(top_dims) * 0.3), 1 + 0.5 * len(physical_names)))
    im = ax.imshow(sub, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    ax.set_yticks(range(len(physical_names))); ax.set_yticklabels(physical_names)
    ax.set_xticks(range(len(top_dims))); ax.set_xticklabels(top_dims, rotation=90, fontsize=7)
    ax.set_xlabel("latent dimension")
    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02); cbar.set_label("Pearson r")
    plt.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.show()
    return top_dims


def top_correlated_dims_per_quantity(corr_matrix, physical_names, top_n=5):
    summary = {}
    for i, name in enumerate(physical_names):
        row = corr_matrix[i]
        order = np.argsort(np.abs(row))[::-1][:top_n]
        summary[name] = [(int(d), float(row[d])) for d in order]
    return summary


def print_correlation_summary(summary):
    for name, items in summary.items():
        items_str = ", ".join(f"#{d} (r={r:+.2f})" for d, r in items)
        print(f"{name:15s} -> {items_str}")


def classify_latent_feature(latent_map, classical_maps, corr_threshold=0.35):
    flat_latent = latent_map.ravel()
    corrs = {name: np.corrcoef(flat_latent, arr.ravel())[0, 1] for name, arr in classical_maps.items()}
    best_name = max(corrs, key=lambda k: abs(corrs[k]))
    best_corr = corrs[best_name]
    if abs(best_corr) < corr_threshold:
        label = "fine-scale / precipitate?"
    elif best_name == "virtual_BF":
        label = "orientation / thickness-sensitive?"
    elif best_name in ("COM_y", "COM_x"):
        label = "strain / lattice-tilt-sensitive?"
    elif best_name == "sharpness":
        label = "disorder / dislocation-sensitive?"
    else:
        label = "uncharacterized?"
    return label, best_name, best_corr, corrs


In [ ]:
for d in registry:
    name = d["name"]
    print(f"\n=== {name}: latent-physical correlations ===")
    corr_matrix, physical_names = compute_latent_physical_correlations(
        results[name]["latent_maps"], results[name]["classical_maps"])
    plot_correlation_heatmap(corr_matrix, physical_names,
                              out_path=os.path.join(OUT_DIR_BASE, name, "correlation_heatmap.png"))
    summary = top_correlated_dims_per_quantity(corr_matrix, physical_names, top_n=5)
    print_correlation_summary(summary)

    print(f"\n{name} feature classification (top diverse dims):")
    for dim in results[name]["top_dims"]:
        label, best_name, best_corr, _ = classify_latent_feature(
            results[name]["latent_maps"][:, :, dim], results[name]["classical_maps"])
        print(f"  latent #{dim}: {label}  (best match: {best_name}, r={best_corr:.2f})")


In [ ]:
def make_publication_figure(panels, out_path, step_size_um=None, scalebar_frac=0.2,
                             ncols=None, panel_height=3.2, dpi=300, cmap="gray"):
    n = len(panels)
    ncols = ncols or n
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_height * ncols, panel_height * nrows))
    axes = np.atleast_1d(axes).ravel()
    letters = string.ascii_uppercase
    for i, panel in enumerate(panels):
        ax = axes[i]
        data = panel["data"]
        vmin = panel.get("vmin", np.percentile(data, 1))
        vmax = panel.get("vmax", np.percentile(data, 99))
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(panel.get("title", ""), fontsize=10)
        ax.axis("off")
        ax.text(0.03, 0.95, letters[i], transform=ax.transAxes, fontsize=13, fontweight="bold",
                 color="white", ha="left", va="top",
                 bbox=dict(boxstyle="square,pad=0.15", facecolor="black", alpha=0.6, edgecolor="none"))
        if panel.get("colorbar", False):
            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            if "cbar_label" in panel:
                cbar.set_label(panel["cbar_label"], fontsize=9)
            cbar.ax.tick_params(labelsize=8)
        if i == 0 and step_size_um is not None:
            dy, dx = data.shape
            bar_um = round(dx * step_size_um * scalebar_frac, -1) or 10
            bar_px = bar_um / step_size_um
            x0, y0 = dx * 0.06, dy * 0.92
            ax.plot([x0, x0 + bar_px], [y0, y0], color="white", linewidth=3, solid_capstyle="butt")
            ax.text(x0 + bar_px / 2, y0 - dy * 0.03, f"{bar_um:.0f} \u00b5m",
                     color="white", ha="center", va="bottom", fontsize=9)
    for j in range(len(panels), len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out_path}")


STEP_SIZE_UM = None   # set to your real scan step size (um) per dataset if known

for d in registry:
    name = d["name"]
    latent_maps = results[name]["latent_maps"]
    classical_maps = results[name]["classical_maps"]
    top_dims = results[name]["top_dims"]

    panels = [{"title": f"latent #{dim}", "data": latent_maps[:, :, dim]} for dim in top_dims]
    panels.append({"title": "virtual_BF", "data": classical_maps["virtual_BF"]})
    panels.append({"title": "COM_y", "data": classical_maps["COM_y"], "colorbar": True, "cbar_label": "COM shift (px)"})
    panels.append({"title": "COM_x", "data": classical_maps["COM_x"], "colorbar": True, "cbar_label": "COM shift (px)"})
    panels.append({"title": "sharpness", "data": classical_maps["sharpness"]})

    make_publication_figure(panels, out_path=os.path.join(OUT_DIR_BASE, name, "publication_figure.png"),
                             step_size_um=STEP_SIZE_UM, ncols=len(panels))


In [ ]:
def export_maps_for_dm(latent_maps, top_dims, classical_maps, out_dir, step_size_um=None,
                        latent_labels=None, dtype="float32"):
    export_dir = os.path.join(out_dir, "for_DM")
    os.makedirs(export_dir, exist_ok=True)
    resolution_kwargs = {}
    if step_size_um is not None:
        px_per_cm = 1e4 / step_size_um
        resolution_kwargs = {"resolution": (px_per_cm, px_per_cm),
                              "resolutionunit": tifffile.RESUNIT.CENTIMETER,
                              "metadata": {"unit": "um", "pixel_size_um": step_size_um}}
    saved_paths = []
    for dim in top_dims:
        arr = latent_maps[:, :, dim].astype(dtype)
        label = (latent_labels or {}).get(dim, "")
        suffix = f"_{label.replace(' ', '_').replace('/', '-')}" if label else ""
        path = os.path.join(export_dir, f"latent_{dim:03d}{suffix}.tiff")
        tifffile.imwrite(path, arr, **resolution_kwargs)
        saved_paths.append(path)
    for map_name, arr in classical_maps.items():
        path = os.path.join(export_dir, f"classical_{map_name}.tiff")
        tifffile.imwrite(path, arr.astype(dtype), **resolution_kwargs)
        saved_paths.append(path)
    print(f"Exported {len(saved_paths)} calibrated TIFFs to {export_dir}")
    return saved_paths


def export_full_scan_overview_for_dm(vbf, vadf, out_dir, step_size_um=None, dtype="float32"):
    export_dir = os.path.join(out_dir, "for_DM")
    os.makedirs(export_dir, exist_ok=True)
    resolution_kwargs = {}
    if step_size_um is not None:
        px_per_cm = 1e4 / step_size_um
        resolution_kwargs = {"resolution": (px_per_cm, px_per_cm),
                              "resolutionunit": tifffile.RESUNIT.CENTIMETER,
                              "metadata": {"unit": "um", "pixel_size_um": step_size_um}}
    paths = {}
    for map_name, arr in [("complete_virtual_BF", vbf), ("complete_virtual_ADF", vadf)]:
        path = os.path.join(export_dir, f"{map_name}.tiff")
        tifffile.imwrite(path, arr.astype(dtype), **resolution_kwargs)
        paths[map_name] = path
        print(f"Saved {path}")
    return paths


In [ ]:
for d in registry:
    name = d["name"]
    out_dir = os.path.join(OUT_DIR_BASE, name)
    latent_labels = {dim: classify_latent_feature(results[name]["latent_maps"][:, :, dim],
                                                    results[name]["classical_maps"])[0]
                      for dim in results[name]["top_dims"]}
    export_maps_for_dm(results[name]["latent_maps"], results[name]["top_dims"],
                        results[name]["classical_maps"], out_dir,
                        step_size_um=STEP_SIZE_UM, latent_labels=latent_labels)
    export_full_scan_overview_for_dm(results[name]["vbf_quicklook"], results[name]["vadf_quicklook"],
                                       out_dir, step_size_um=STEP_SIZE_UM)


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def choose_n_clusters(latent_maps, dims=None, k_range=range(2, 9), sample_size=3000, random_state=0):
    ny, nx, latent_dim = latent_maps.shape
    flat = latent_maps[:, :, dims].reshape(-1, len(dims)) if dims is not None else latent_maps.reshape(-1, latent_dim)
    rng = np.random.default_rng(random_state)
    idx = rng.choice(len(flat), size=min(sample_size, len(flat)), replace=False)
    sample = flat[idx]
    scores = {}
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=5)
        labels = km.fit_predict(sample)
        scores[k] = silhouette_score(sample, labels)
    return max(scores, key=scores.get), scores


def segment_latent_space(latent_maps, n_clusters, dims=None, random_state=0):
    ny, nx, latent_dim = latent_maps.shape
    flat = latent_maps[:, :, dims].reshape(-1, len(dims)) if dims is not None else latent_maps.reshape(-1, latent_dim)
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labels = km.fit_predict(flat)
    return labels.reshape(ny, nx), km


def plot_segmentation(segmentation_map, out_path=None, n_clusters=None, dpi=200):
    n_clusters = n_clusters or (segmentation_map.max() + 1)
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(segmentation_map, cmap="tab10", vmin=0, vmax=max(9, n_clusters - 1))
    ax.set_title(f"Unsupervised segmentation ({n_clusters} clusters)")
    ax.axis("off")
    cbar = fig.colorbar(im, ax=ax, ticks=range(n_clusters), fraction=0.046)
    cbar.set_label("cluster ID")
    plt.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.show()


In [ ]:
PRIMARY_NAME = registry[0]["name"]
latent_maps = results[PRIMARY_NAME]["latent_maps"]

best_k, k_scores = choose_n_clusters(latent_maps)
print("Silhouette scores per k:", {k: round(v, 3) for k, v in k_scores.items()})
print(f"Suggested cluster count: {best_k}")

seg_map, km = segment_latent_space(latent_maps, n_clusters=best_k)
plot_segmentation(seg_map, out_path=os.path.join(OUT_DIR_BASE, PRIMARY_NAME, "latent_segmentation.png"), n_clusters=best_k)
np.save(os.path.join(OUT_DIR_BASE, PRIMARY_NAME, "segmentation_map.npy"), seg_map)


In [ ]:
def crop_maps(latent_maps, classical_maps, roi):
    y0, y1, x0, x1 = roi
    cropped_latent = latent_maps[y0:y1, x0:x1, :]
    cropped_classical = {name: arr[y0:y1, x0:x1] for name, arr in classical_maps.items()}
    return cropped_latent, cropped_classical


def make_roi_comparison_figure(latent_maps, classical_maps, rois, roi_labels, out_path,
                                 top_k=6, corr_threshold=0.75, panel_height=2.8, dpi=300):
    n_rois = len(rois)
    fig, axes = plt.subplots(n_rois, top_k, figsize=(panel_height * top_k, panel_height * n_rois))
    if n_rois == 1:
        axes = axes.reshape(1, -1)
    row_letters = string.ascii_uppercase

    per_roi_dims = []
    for r, (roi, label) in enumerate(zip(rois, roi_labels)):
        cropped_latent, cropped_classical = crop_maps(latent_maps, classical_maps, roi)
        top_dims, top_vars = select_diverse_latent_dims(cropped_latent, top_k=top_k, corr_threshold=corr_threshold)
        per_roi_dims.append(top_dims)

        for c in range(top_k):
            ax = axes[r, c]
            if c < len(top_dims):
                dim = top_dims[c]
                data = cropped_latent[:, :, dim]
                vmin, vmax = np.percentile(data, 1), np.percentile(data, 99)
                ax.imshow(data, cmap="gray", vmin=vmin, vmax=vmax)
                label_text = f"{row_letters[r]}.{c+1} - #{dim}" if c == 0 else f"{row_letters[r]}.{c+1}"
                ax.text(0.03, 0.95, label_text, transform=ax.transAxes, fontsize=9, fontweight="bold",
                         color="white", ha="left", va="top",
                         bbox=dict(boxstyle="square,pad=0.15", facecolor="black", alpha=0.8, edgecolor="none"))
                if c == 0:
                    ax.text(0.03, 0.03, label, transform=ax.transAxes, fontsize=8,
                             color="white", ha="left", va="bottom", fontweight="bold")
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

    plt.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.show()
    return per_roi_dims


In [ ]:
ny, nx = latent_maps.shape[:2]
ROI_1 = (0, ny // 2, 0, nx // 2)          # change to your region of interest
ROI_2 = (ny // 2, ny, nx // 2, nx)        # change to your region of interest

per_roi_dims = make_roi_comparison_figure(
    latent_maps, results[PRIMARY_NAME]["classical_maps"],
    rois=[ROI_1, ROI_2], roi_labels=["Region 1", "Region 2"],
    out_path=os.path.join(OUT_DIR_BASE, PRIMARY_NAME, "roi_comparison_figure.png"),
    top_k=6,
)
print("Diverse dims found per region:", [d.tolist() for d in per_roi_dims])


In [ ]:
import pandas as pd

def build_architecture_table(in_size, latent_dim, base_ch, kernel):
    m = DiffractionVAE(in_size=in_size, latent_dim=latent_dim, base_ch=base_ch, kernel=kernel)
    rows = []

    def count_params(mod):
        return sum(p.numel() for p in mod.parameters())

    x = torch.randn(1, 1, in_size, in_size)
    b1 = m.encoder.block1(x); b2 = m.encoder.block2(b1); b3 = m.encoder.block3(b2)

    rows.append(["Encoder", "Input", f"{in_size}x{in_size}", 1, "-", 0])
    rows.append(["Encoder", "Block 1 (conv+res)", f"{b1.shape[-1]}x{b1.shape[-1]}", base_ch, kernel, count_params(m.encoder.block1)])
    rows.append(["Encoder", "Block 2 (conv+res)", f"{b2.shape[-1]}x{b2.shape[-1]}", base_ch*2, kernel, count_params(m.encoder.block2)])
    rows.append(["Encoder", "Block 3 (conv+res)", f"{b3.shape[-1]}x{b3.shape[-1]}", base_ch*4, kernel, count_params(m.encoder.block3)])
    rows.append(["Encoder", "Global avg pool", "1x1", base_ch*4, "-", 0])
    rows.append(["Encoder", "FC (mu, logvar)", "-", latent_dim, "-", count_params(m.encoder.fc_mu)+count_params(m.encoder.fc_logvar)])

    z = torch.randn(1, latent_dim)
    d0 = m.decoder.fc(z).view(-1, m.decoder.base_ch, m.decoder.init_size, m.decoder.init_size)
    d1 = m.decoder.block1(d0); d2 = m.decoder.block2(d1); d3 = m.decoder.block3(d2)
    out = m.decoder.out_act(m.decoder.out_conv(d3))

    rows.append(["Decoder", "FC + reshape", f"{d0.shape[-1]}x{d0.shape[-1]}", m.decoder.base_ch, "-", count_params(m.decoder.fc)])
    rows.append(["Decoder", "Block 1 (deconv+res)", f"{d1.shape[-1]}x{d1.shape[-1]}", base_ch*2, kernel, count_params(m.decoder.block1)])
    rows.append(["Decoder", "Block 2 (deconv+res)", f"{d2.shape[-1]}x{d2.shape[-1]}", base_ch, kernel, count_params(m.decoder.block2)])
    rows.append(["Decoder", "Block 3 (deconv+res)", f"{d3.shape[-1]}x{d3.shape[-1]}", base_ch, kernel, count_params(m.decoder.block3)])
    rows.append(["Decoder", "Output conv (1x1) + tanh", f"{out.shape[-1]}x{out.shape[-1]}", 1, 1, count_params(m.decoder.out_conv)])

    df = pd.DataFrame(rows, columns=["Stage", "Layer", "Output size", "Channels", "Kernel", "Parameters"])
    return df, sum(p.numel() for p in m.parameters())


arch_df, arch_total_params = build_architecture_table(in_size=BINNED_SIZE, latent_dim=LATENT_DIM, base_ch=BASE_CH, kernel=KERNEL_SIZE)
print(arch_df.to_string(index=False))
print(f"\nTotal parameters: {arch_total_params:,}")

arch_df.to_csv(os.path.join(OUT_DIR_BASE, "architecture_table.csv"), index=False)
with open(os.path.join(OUT_DIR_BASE, "architecture_table.md"), "w") as f:
    f.write(arch_df.to_markdown(index=False))
print(f"Saved architecture_table.csv / .md to {OUT_DIR_BASE}")


In [ ]:
def evaluate_reconstruction_quality(model, val_loader, active_mask_t, device=DEVICE):
    model.eval()
    correlations, nrmses = [], []
    mask_np = active_mask_t.cpu().numpy().astype(bool)
    with torch.no_grad():
        for x, _, _ in val_loader:
            x = x.to(device)
            recon, mu, logvar, z = model(x)
            x_np = x.cpu().numpy()
            recon_np = recon.cpu().numpy()
            for i in range(x_np.shape[0]):
                orig = x_np[i, 0][mask_np]
                rec = recon_np[i, 0][mask_np]
                corr = np.corrcoef(orig, rec)[0, 1] if orig.std() > 1e-8 and rec.std() > 1e-8 else np.nan
                rmse = np.sqrt(np.mean((orig - rec) ** 2))
                nrmse = rmse / (orig.max() - orig.min() + 1e-8)
                correlations.append(corr)
                nrmses.append(nrmse)
    correlations = np.array(correlations)
    nrmses = np.array(nrmses)
    return {
        "correlation_p10": np.nanpercentile(correlations, 10),
        "correlation_p50": np.nanpercentile(correlations, 50),
        "correlation_p90": np.nanpercentile(correlations, 90),
        "nrmse_p10": np.percentile(nrmses, 10),
        "nrmse_p50": np.percentile(nrmses, 50),
        "nrmse_p90": np.percentile(nrmses, 90),
        "n_samples": len(correlations),
    }


def build_performance_table(history, eval_metrics, n_diverse_dims, total_params, training_time_sec=None):
    row = {
        "Latent dimension": LATENT_DIM,
        "Total parameters": f"{total_params:,}",
        "Epochs trained": len(history["train_l2"]),
        "Final train L2 loss": f"{history['train_l2'][-1]:.2f}",
        "Final val L2 loss": f"{history['val_l2'][-1]:.2f}",
        "Final KL divergence": f"{history['train_kld'][-1]:.2f}",
        "Final latent mu std": f"{history['mu_std'][-1]:.3f}",
        "Diverse (independent) latent dims found": n_diverse_dims,
        "Reconstruction correlation (P10/P50/P90)": f"{eval_metrics['correlation_p10']:.3f} / {eval_metrics['correlation_p50']:.3f} / {eval_metrics['correlation_p90']:.3f}",
        "Normalized RMSE (P10/P50/P90)": f"{eval_metrics['nrmse_p10']:.3f} / {eval_metrics['nrmse_p50']:.3f} / {eval_metrics['nrmse_p90']:.3f}",
        "Validation samples evaluated": eval_metrics["n_samples"],
    }
    if training_time_sec is not None:
        row["Training time"] = f"{training_time_sec/60:.1f} min"
    return pd.DataFrame(list(row.items()), columns=["Metric", "Value"])


In [ ]:
eval_metrics = evaluate_reconstruction_quality(model, val_loader, active_mask_t)
n_diverse_whole_map = len(results[PRIMARY_NAME]["top_dims"])

perf_df = build_performance_table(history, eval_metrics, n_diverse_whole_map, arch_total_params,
                                    training_time_sec=TRAINING_TIME_SEC if "TRAINING_TIME_SEC" in dir() else None)
print(perf_df.to_string(index=False))

perf_df.to_csv(os.path.join(OUT_DIR_BASE, "performance_table.csv"), index=False)
with open(os.path.join(OUT_DIR_BASE, "performance_table.md"), "w") as f:
    f.write(perf_df.to_markdown(index=False))
print(f"Saved performance_table.csv / .md to {OUT_DIR_BASE}")


In [ ]:
from sklearn.decomposition import PCA as _PCA_grad

def compute_dominant_gradient(latent_maps, dims=None):
    ny, nx, latent_dim = latent_maps.shape
    flat = latent_maps[:, :, dims].reshape(-1, len(dims)) if dims is not None else latent_maps.reshape(-1, latent_dim)
    pca = _PCA_grad(n_components=1, random_state=0)
    pc1 = pca.fit_transform(flat)[:, 0]
    return pc1.reshape(ny, nx), pca


def diagnose_gradient_orientation(map_2d):
    """Correlates a 2D map against scan row and column index separately"""
    ny, nx = map_2d.shape
    iy, ix = np.mgrid[0:ny, 0:nx]
    r_row = np.corrcoef(map_2d.ravel(), iy.ravel())[0, 1]
    r_col = np.corrcoef(map_2d.ravel(), ix.ravel())[0, 1]
    print(f"Correlation with scan row (iy): {r_row:+.3f}")
    print(f"Correlation with scan col (ix): {r_col:+.3f}")
    return r_row, r_col


def fit_and_subtract_plane(map_2d):
    ny, nx = map_2d.shape
    iy, ix = np.mgrid[0:ny, 0:nx]
    A = np.stack([np.ones(ny * nx), iy.ravel(), ix.ravel()], axis=1)
    coef, *_ = np.linalg.lstsq(A, map_2d.ravel(), rcond=None)
    plane = (A @ coef).reshape(ny, nx)
    residual = map_2d - plane
    r2 = 1 - np.sum((map_2d - plane) ** 2) / np.sum((map_2d - map_2d.mean()) ** 2)
    print(f"Linear-plane fit R^2 = {r2:.3f} (close to 1 => consistent with a thickness wedge)")
    return plane, residual, r2


PRIMARY_NAME = registry[0]["name"]
latent_maps = results[PRIMARY_NAME]["latent_maps"]
classical_maps = results[PRIMARY_NAME]["classical_maps"]

gradient_map, gradient_pca = compute_dominant_gradient(latent_maps)

print("Dominant latent gradient (PC1 of full latent space)")
diagnose_gradient_orientation(gradient_map)
plane, plane_residual, plane_r2 = fit_and_subtract_plane(gradient_map)

print("\n virtual_BF, for comparison")
diagnose_gradient_orientation(classical_maps["virtual_BF"])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
robust_imshow(axes[0], gradient_map, cmap="coolwarm", title="Dominant latent gradient (PC1)")
robust_imshow(axes[1], plane, cmap="coolwarm", title=f"Best-fit plane (R\u00b2={plane_r2:.2f})")
robust_imshow(axes[2], plane_residual, cmap="coolwarm", title="Residual after plane subtraction")
robust_imshow(axes[3], classical_maps["virtual_BF"], title="virtual_BF (for comparison)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR_BASE, PRIMARY_NAME, "dominant_gradient_diagnostics.png"), dpi=200)
plt.show()

In [ ]:
def select_diverse_latent_dims_residual(latent_maps, gradient_map, top_k=6, corr_threshold=0.6, candidate_pool=60):
    ny, nx, latent_dim = latent_maps.shape
    flat = latent_maps.reshape(-1, latent_dim)
    g = gradient_map.ravel()
    g_design = np.stack([g, np.ones_like(g)], axis=1)

    residuals = np.zeros_like(flat)
    for d in range(latent_dim):
        coef, *_ = np.linalg.lstsq(g_design, flat[:, d], rcond=None)
        residuals[:, d] = flat[:, d] - g_design @ coef

    residual_variances = residuals.var(axis=0)
    order = np.argsort(residual_variances)[::-1][:candidate_pool]
    selected = []
    for dim in order:
        if len(selected) >= top_k:
            break
        if all(abs(np.corrcoef(residuals[:, dim], residuals[:, s])[0, 1]) <= corr_threshold for s in selected):
            selected.append(dim)
    return np.array(selected), residual_variances[selected], residuals.reshape(ny, nx, latent_dim)


top_dims_residual, top_residual_vars, latent_residuals = select_diverse_latent_dims_residual(
    latent_maps, gradient_map, top_k=6, corr_threshold=0.6)
print(f"Residual-reranked diverse dims: {top_dims_residual.tolist()}")
print(f"(compare to raw-variance dims from earlier: {results[PRIMARY_NAME]['top_dims'].tolist()})")

fig, axes = plt.subplots(1, len(top_dims_residual) + 4, figsize=(3 * (len(top_dims_residual) + 4), 3))
for i, dim in enumerate(top_dims_residual):
    robust_imshow(axes[i], latent_maps[:, :, dim], title=f"latent #{dim}\n(residual-ranked)")
for j, (map_name, arr) in enumerate(classical_maps.items()):
    robust_imshow(axes[len(top_dims_residual) + j], arr, title=map_name)
fig.suptitle(f"{PRIMARY_NAME} -- after removing the dominant gradient")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR_BASE, PRIMARY_NAME, "comparison_figure_residual_ranked.png"), dpi=200)
plt.show()

In [ ]:
@torch.no_grad()
def latent_traversal(model, latent_maps, dims, n_steps=7, span_std=3.0, device=DEVICE):
    model.eval()
    flat = latent_maps.reshape(-1, latent_maps.shape[-1])
    mean_vec_raw = flat.mean(axis=0)
    dists = np.linalg.norm(flat - mean_vec_raw, axis=1)
    mean_vec = flat[np.argmin(dists)]   
    std_vec = flat.std(axis=0)

    traversals = {}
    for dim in dims:
        dim = int(dim)
        steps = np.linspace(-span_std, span_std, n_steps)
        z_batch = np.tile(mean_vec, (n_steps, 1)).astype(np.float32)
        z_batch[:, dim] = mean_vec[dim] + steps * std_vec[dim]
        z_t = torch.from_numpy(z_batch).to(device)
        recon = model.decoder(z_t).cpu().numpy()[:, 0]
        traversals[dim] = {"steps": steps, "images": recon}
    return traversals


def plot_latent_traversals(traversals, out_path, dpi=200):
    dims = list(traversals.keys())
    n_steps = len(next(iter(traversals.values()))["steps"])
    fig, axes = plt.subplots(len(dims), n_steps, figsize=(1.6 * n_steps, 1.6 * len(dims)))
    axes = np.atleast_2d(axes)
    for r, dim in enumerate(dims):
        images = traversals[dim]["images"]
        steps = traversals[dim]["steps"]
        for c in range(n_steps):
            ax = axes[r, c]
            ax.imshow(images[c], cmap="gray")
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(f"{steps[c]:+.1f}\u03c3", fontsize=8)
            if c == 0:
                ax.set_ylabel(f"dim #{dim}", fontsize=9, rotation=0, ha="right", va="center")
    plt.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.show()


traversal_dims = top_dims_residual[:4]
traversals = latent_traversal(model, latent_maps, traversal_dims, n_steps=7, span_std=3.0)
plot_latent_traversals(traversals, out_path=os.path.join(OUT_DIR_BASE, PRIMARY_NAME, "latent_traversals.png"))

In [ ]:
def make_rgb_composite(latent_maps, dims, clip_percentile=1.0):
    assert 1 <= len(dims) <= 3, "RGB composite needs 1-3 dimensions"
    channels = []
    for dim in dims:
        arr = latent_maps[:, :, int(dim)].astype(np.float32)
        lo, hi = np.percentile(arr, clip_percentile), np.percentile(arr, 100 - clip_percentile)
        arr = np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)
        channels.append(arr)
    while len(channels) < 3:
        channels.append(np.zeros_like(channels[0]))
    return np.stack(channels, axis=-1)


def plot_rgb_composite(rgb, dims, out_path, dpi=200):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(rgb)
    ax.axis("off")
    labels = ["R", "G", "B"]
    legend_text = "  ".join(f"{labels[i]}=latent #{int(dim)}" for i, dim in enumerate(dims))
    ax.set_title(f"Composite latent map ({legend_text})", fontsize=10)
    plt.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.show()


dims_for_rgb = top_dims_residual[:3]
rgb = make_rgb_composite(latent_maps, dims_for_rgb)
plot_rgb_composite(rgb, dims_for_rgb, out_path=os.path.join(OUT_DIR_BASE, PRIMARY_NAME, "rgb_composite_latent_residual.png"))

In [ ]:
pip install umap

In [ ]:
try:
    import umap
    HAVE_UMAP = True
except ImportError:
    HAVE_UMAP = False
    print("umap-learn not installed -- falling back to PCA.")

from sklearn.decomposition import PCA


def embed_latent_space(latent_maps, dims=None, n_sample=5000, random_state=0):
    ny, nx, latent_dim = latent_maps.shape
    flat = latent_maps[:, :, dims].reshape(-1, len(dims)) if dims is not None else latent_maps.reshape(-1, latent_dim)
    rng = np.random.default_rng(random_state)
    idx = rng.choice(len(flat), size=min(n_sample, len(flat)), replace=False)
    sample = flat[idx]
    reducer = umap.UMAP(n_components=2, random_state=random_state) if HAVE_UMAP else PCA(n_components=2, random_state=random_state)
    embedding = reducer.fit_transform(sample)
    return embedding, idx


def plot_latent_embedding(embedding, color_values, color_label, out_path, dpi=200, cmap="viridis"):
    fig, ax = plt.subplots(figsize=(6, 5))
    sc = ax.scatter(embedding[:, 0], embedding[:, 1], c=color_values, cmap=cmap, s=6, alpha=0.7)
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(color_label)
    method = "UMAP" if HAVE_UMAP else "PCA"
    ax.set_title(f"Latent-space structure ({method}), colored by {color_label}")
    ax.set_xlabel(f"{method} 1"); ax.set_ylabel(f"{method} 2")
    plt.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.show()


embedding, idx = embed_latent_space(latent_maps, dims=None, n_sample=5000)
plot_latent_embedding(embedding, classical_maps["virtual_BF"].ravel()[idx], "virtual_BF (a.u.)",
                      out_path=os.path.join(OUT_DIR_BASE, PRIMARY_NAME, "latent_embedding_by_BF.png"))

In [ ]:
@torch.no_grad()
def compute_residual_maps(model, mib_reader, background, active_mask, device=DEVICE):
    model.eval()
    ny, nx = mib_reader.ny, mib_reader.nx
    mask_np = active_mask.astype(bool)
    residual_rmse = np.zeros((ny, nx), dtype=np.float32)
    residual_signed_mean = np.zeros((ny, nx), dtype=np.float32)
    t0 = time.time()
    for iy in range(ny):
        row_raw = mib_reader.get_row(iy)
        row_proc = np.stack([preprocess_pattern(row_raw[ix], background, active_mask=active_mask) for ix in range(nx)])
        x = torch.from_numpy(row_proc).unsqueeze(1).to(device)
        recon, mu, logvar, z = model(x)
        diff = (recon[:, 0].cpu().numpy() - row_proc)
        for ix in range(nx):
            dpix = diff[ix][mask_np]
            residual_rmse[iy, ix] = np.sqrt(np.mean(dpix ** 2))
            residual_signed_mean[iy, ix] = dpix.mean()
        if iy % 32 == 0:
            print(f"    residual row {iy}/{ny}  ({time.time()-t0:.0f}s)", flush=True)
    return residual_rmse, residual_signed_mean


def plot_residual_maps(residual_rmse, residual_signed_mean, out_path, dpi=200):
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    im0 = axes[0].imshow(residual_rmse, cmap="magma")
    axes[0].set_title("Reconstruction RMSE per position"); axes[0].axis("off")
    fig.colorbar(im0, ax=axes[0], fraction=0.046)
    vmax = np.percentile(np.abs(residual_signed_mean), 99)
    im1 = axes[1].imshow(residual_signed_mean, cmap="coolwarm", vmin=-vmax, vmax=vmax)
    axes[1].set_title("Signed mean residual\n(systematic bias / drift)"); axes[1].axis("off")
    fig.colorbar(im1, ax=axes[1], fraction=0.046)
    plt.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.show()


residual_rmse, residual_signed_mean = compute_residual_maps(model, readers[PRIMARY_NAME], backgrounds[PRIMARY_NAME], active_mask)
np.save(os.path.join(OUT_DIR_BASE, PRIMARY_NAME, "residual_rmse.npy"), residual_rmse)
np.save(os.path.join(OUT_DIR_BASE, PRIMARY_NAME, "residual_signed_mean.npy"), residual_signed_mean)
plot_residual_maps(residual_rmse, residual_signed_mean, out_path=os.path.join(OUT_DIR_BASE, PRIMARY_NAME, "residual_maps.png"))

In [ ]:
def encoder_saliency_smoothgrad(model, pattern, dim, device=DEVICE, n_samples=25, noise_std=0.05):
    model.eval()
    sal_accum = np.zeros_like(pattern, dtype=np.float64)
    for _ in range(n_samples):
        noisy = pattern + np.random.normal(0, noise_std, size=pattern.shape).astype(np.float32)
        x = torch.from_numpy(noisy).unsqueeze(0).unsqueeze(0).to(device).requires_grad_(True)
        mu, logvar = model.encoder(x)
        mu[:, dim].sum().backward()
        sal_accum += x.grad[0, 0].abs().cpu().numpy()
    return sal_accum / n_samples


def plot_saliency_grid_smoothgrad(model, pattern, dims, out_path, device=DEVICE, n_samples=25, noise_std=0.05, dpi=200):
    fig, axes = plt.subplots(2, len(dims), figsize=(2.8 * len(dims), 5.6))
    if len(dims) == 1:
        axes = axes.reshape(2, 1)
    for c, dim in enumerate(dims):
        dim = int(dim)
        sal = encoder_saliency_smoothgrad(model, pattern, dim, device=device, n_samples=n_samples, noise_std=noise_std)
        axes[0, c].imshow(pattern, cmap="gray")
        axes[0, c].set_title(f"input (for dim #{dim})", fontsize=8); axes[0, c].axis("off")
        axes[1, c].imshow(sal, cmap="inferno", vmax=np.percentile(sal, 99.5))
        axes[1, c].set_title(f"SmoothGrad saliency: dim #{dim}", fontsize=8); axes[1, c].axis("off")
    plt.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.show()


r = readers[PRIMARY_NAME]
example_raw = r.get_frame(r.ny // 2, r.nx // 2)
example_pattern = preprocess_pattern(example_raw, backgrounds[PRIMARY_NAME], active_mask=active_mask)
saliency_dims = top_dims_residual[:4]
plot_saliency_grid_smoothgrad(model, example_pattern, saliency_dims,
                                out_path=os.path.join(OUT_DIR_BASE, PRIMARY_NAME, "encoder_saliency_smoothgrad.png"))

In [ ]:
LATENT_CMAP = "coolwarm"
CLASSICAL_CMAP = "gray"
STEP_SIZE_UM = None   # set to your real scan step size (um) if known


def add_scalebar(ax, arr_shape, step_size_um, frac=0.2, color="white"):
    if step_size_um is None:
        return
    dy, dx = arr_shape
    bar_um = round(dx * step_size_um * frac, -1) or 10
    bar_px = bar_um / step_size_um
    x0, y0 = dx * 0.06, dy * 0.92
    ax.plot([x0, x0 + bar_px], [y0, y0], color=color, linewidth=3, solid_capstyle="butt")
    ax.text(x0 + bar_px / 2, y0 - dy * 0.03, f"{bar_um:.0f} \u00b5m", color=color, ha="center", va="bottom", fontsize=9)


def make_publication_figure_v2(panels, out_path, step_size_um=None, ncols=None, panel_height=3.2, dpi=300):
    n = len(panels)
    ncols = ncols or n
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_height * ncols, panel_height * nrows))
    axes = np.atleast_1d(axes).ravel()
    letters = string.ascii_uppercase
    for i, panel in enumerate(panels):
        ax = axes[i]
        data = panel["data"]
        is_latent = panel.get("is_latent", False)
        cmap = panel.get("cmap", LATENT_CMAP if is_latent else CLASSICAL_CMAP)
        if is_latent:
            vmax = np.percentile(np.abs(data), 99)
            vmin = -vmax
        else:
            vmin = panel.get("vmin", np.percentile(data, 1))
            vmax = panel.get("vmax", np.percentile(data, 99))
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(panel.get("title", ""), fontsize=10)
        ax.axis("off")
        label_color = "black" if is_latent else "white"
        box_color = "white" if is_latent else "black"
        ax.text(0.03, 0.95, letters[i], transform=ax.transAxes, fontsize=13, fontweight="bold",
                 color=label_color, ha="left", va="top",
                 bbox=dict(boxstyle="square,pad=0.15", facecolor=box_color, alpha=0.65, edgecolor="none"))
        if panel.get("colorbar", True):
            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            if "cbar_label" in panel:
                cbar.set_label(panel["cbar_label"], fontsize=9)
            cbar.ax.tick_params(labelsize=8)
        add_scalebar(ax, data.shape, step_size_um, color=label_color)
    for j in range(len(panels), len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out_path}")


panels = [{"title": f"latent #{dim}", "data": latent_maps[:, :, dim], "is_latent": True} for dim in top_dims_residual]
panels.append({"title": "virtual_BF", "data": classical_maps["virtual_BF"]})
panels.append({"title": "COM_y", "data": classical_maps["COM_y"], "cbar_label": "COM shift (px)"})
panels.append({"title": "COM_x", "data": classical_maps["COM_x"], "cbar_label": "COM shift (px)"})
panels.append({"title": "sharpness", "data": classical_maps["sharpness"]})

make_publication_figure_v2(panels, out_path=os.path.join(OUT_DIR_BASE, PRIMARY_NAME, "residual_ranked.png"),
                            step_size_um=STEP_SIZE_UM, ncols=len(panels))